# 05 — Dropout

A network with spare capacity memorizes its training set. Chapter 11 met two cures for that — an L2
penalty on the weights, and early stopping. This notebook builds a third, one invented for neural
networks: **dropout**. The idea is almost suspiciously simple — randomly switch off half the neurons
on every training step — and yet it regularizes by turning one network into an *implicit ensemble* of
many. We build it by hand (scikit-learn's `MLPClassifier` has no dropout at all), see exactly what it
does to the signal, and measure — honestly — how much it helps.

**Prerequisites**
- Chapter 00 (NB 9–10) — over/underfitting and the train/validation gap.
- Chapter 03 / chapter 11 (NB 4) — L2 / `alpha` and early stopping (the regularizers we compare to).
- Notebooks 1–4 — the `L`-layer network (forward, backward, training loop) and He initialization.

**What you'll be able to do**
- Implement **inverted dropout** and show it preserves the signal's mean while injecting noise.
- Explain *why* it regularizes — the implicit-ensemble / co-adaptation argument.
- Wire it into a network (mask the forward pass *and* the gradient) and measure its effect honestly.
- Place dropout against L2 and early stopping — three different levers on the same problem.

## Another lever on overfitting

When a network has more capacity than the problem needs, it fits the training data perfectly and
generalizes poorly — train accuracy far above validation accuracy. You already have two ways to fight
this: **L2** (add a penalty on large weights, so the network prefers simpler solutions) and **early
stopping** (halt training before it starts memorizing).

**Dropout** is a third lever, specific to neural networks: during training, at every step, randomly
**switch off** a fraction `p` of the hidden units. It sounds destructive — and it is exactly that
destruction that regularizes.

## The mechanism: inverted dropout

On each training step, for each hidden unit independently: **keep** it with probability `1 − p`,
**zero** it with probability `p`. If we stopped there, the layer's output would shrink by a factor
`1 − p` on average, and test time (no dropout) would see signals of a different size than training.

So we rescale the survivors by `1 / (1 − p)` — this is **inverted dropout**. With `p = 0.5`, half the
units vanish and the other half are doubled, so the *expected* activation is unchanged. The payoff:
at **test time we use the whole network with no dropout and no rescale** — training already
compensated, so train and test see signals of the same expected size.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ml_course import colors, viz

viz.use_course_style()

SEED = 0


def inverted_dropout(a, p, rng):
    # Keep each unit with prob (1-p); rescale survivors by 1/(1-p) so E[output] == a.
    mask = (rng.random(a.shape) > p) / (1.0 - p)
    return a * mask, mask


# Check it on a large batch of activations ~ N(0, 1): mean preserved, variance injected.
rng = np.random.default_rng(SEED)
a = rng.standard_normal((100_000, 1))
print(f"original     : mean {a.mean():+.3f}   variance {a.var():.3f}")
for p in (0.2, 0.5, 0.8):
    dropped, mask = inverted_dropout(a, p, rng)
    print(f"p={p}: mean {dropped.mean():+.3f}   variance {dropped.var():.3f}   "
          f"E[mask]={mask.mean():.3f}")

In [ ]:
# A single 20-unit activation vector, before and after dropout (p=0.5).
demo_rng = np.random.default_rng(3)
h = np.abs(demo_rng.standard_normal(20)) + 0.2   # positive activations (post-ReLU)
h_drop, m = inverted_dropout(h, 0.5, demo_rng)
units = np.arange(20)

fig, (axm, axv) = plt.subplots(1, 2, figsize=(14, 4.6))

axm.bar(units - 0.2, h, width=0.4, color=colors.COLORS["muted"], label="before")
axm.bar(units + 0.2, h_drop, width=0.4, color=colors.COLORS["model"], label="after dropout")
axm.set_xlabel("hidden unit")
axm.set_ylabel("activation")
axm.set_title("p = 0.5: half switched off, survivors doubled")
axm.legend()

# Variance before/after on the big N(0,1) batch (p=0.5).
dropped_half, _ = inverted_dropout(a, 0.5, rng)
bars = axv.bar(["before", "after\ndropout"], [a.var(), dropped_half.var()],
               color=[colors.COLORS["muted"], colors.COLORS["model"]],
               edgecolor=colors.COLORS["text"], linewidth=0.6)
for bar, v in zip(bars, [a.var(), dropped_half.var()], strict=True):
    axv.text(bar.get_x() + bar.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom",
             color=colors.COLORS["text"])
axv.set_ylabel("variance")
axv.set_title("Mean preserved, variance ~doubled")
fig.tight_layout()
plt.show()

**Read the figure.** Left: a layer of twenty activations before (grey) and after (blue) dropout at
`p = 0.5`. About half are zeroed; the survivors are scaled up by `×2`, so the layer's average output
is unchanged. Right: across a large batch, the mean stays put while the **variance roughly doubles**
(`1.0 → ~2.0`) — dropout has injected noise without shifting the signal's center. Every training step,
the network sees a **different random sub-network**.

## Why it helps: an implicit ensemble

Switching off a random half of the units each step means each step trains a *different* thinned
network — and with `H` hidden units there are `2^H` such sub-networks, all sharing one set of weights.
Training nudges whichever sub-network appeared on that step. At test time, using the **full** network
(with the inverted-dropout rescale) approximates **averaging the predictions of that whole ensemble**
(more precisely a *geometric* mean — Srivastava et al. §7), and averaging many models is the move that
reduces variance (chapter 06, bagging).

There is a second view of the same fact: because any given partner unit might vanish on the next step,
no unit can **co-adapt** — rely on a specific other unit to fix its mistakes. Each unit is forced to
carry signal that is useful on its own. Robust, non-co-adapted features generalize better.

## Wiring it into the network

Dropout touches two places in the `L`-layer network we already built. In the **forward** pass (training
mode only) we multiply each hidden activation by its dropout mask. In the **backward** pass we gate the
gradient by the *same* mask — a unit that was switched off contributes nothing and receives no gradient.
In **eval** mode we skip both: the full network, no rescale. We reuse our net (He initialization,
notebook 4) and add exactly that.

In [ ]:
def init_params(sizes, seed=SEED):
    # He initialization (notebook 4): std = sqrt(2 / fan_in).
    rng = np.random.default_rng(seed)
    return [{"W": rng.standard_normal((a, b)) * np.sqrt(2.0 / a), "b": np.zeros(b)}
            for a, b in zip(sizes[:-1], sizes[1:], strict=True)]


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def forward(params, X, p_drop=0.0, rng=None, train=False):
    acts, masks, a, last = [X], [None], X, len(params) - 1
    for i, layer in enumerate(params):
        z = a @ layer["W"] + layer["b"]
        if i == last:
            a = softmax(z)              # output layer: never dropped
            m = None
        else:
            a = np.maximum(0.0, z)      # ReLU hidden layer
            if train and p_drop > 0.0:
                m = (rng.random(a.shape) > p_drop) / (1.0 - p_drop)
                a = a * m               # apply dropout, store the POST-mask activation
            else:
                m = None
        acts.append(a)
        masks.append(m)
    return acts, masks


def backward(params, acts, masks, y):
    n = len(y)
    y_onehot = np.zeros_like(acts[-1])
    y_onehot[np.arange(n), y] = 1.0
    d = (acts[-1] - y_onehot) / n
    grads = [None] * len(params)
    for i in reversed(range(len(params))):
        grads[i] = {"W": acts[i].T @ d, "b": d.sum(axis=0)}
        if i > 0:
            da = d @ params[i]["W"].T
            if masks[i] is not None:
                da = da * masks[i]       # gate the gradient by the SAME dropout mask
            d = da * (acts[i] > 0)       # ReLU'
    return grads


print("net ready: forward(train=True) applies dropout; forward() (eval) uses the full network")

## The test: does it reduce overfitting?

Dropout breaks **co-adaptation**, and co-adaptation is worst where units have many partners to lean on
— that is, in **high dimensions**. On a 2-D toy the effect is tiny; so we use a harder setting: 50
features, 10 of them informative, and only **200 training rows** for a wide network with plenty of
capacity to memorize. We train with no regularizer, with dropout, and with L2, and read the held-out
gap.

In [ ]:
X, y_all = make_classification(
    n_samples=2000, n_features=50, n_informative=10, n_redundant=10,
    n_classes=2, class_sep=1.0, random_state=SEED,
)
X = StandardScaler().fit_transform(X)
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y_all, train_size=200, test_size=1000, stratify=y_all, random_state=SEED
)
sizes = [50, 256, 256, 2]   # wide: capacity to memorize 200 rows


def accuracy(params, Xa, ya):
    return (forward(params, Xa)[0][-1].argmax(1) == ya).mean()


def train(sizes, p_drop=0.0, alpha=0.0, lr=0.1, epochs=1500, batch=32, seed=SEED):
    params = init_params(sizes, seed)
    rng = np.random.default_rng(seed)
    n = X_tr.shape[0]
    for _ in range(epochs):
        order = rng.permutation(n)
        for s in range(0, n, batch):
            b = order[s:s + batch]
            acts, masks = forward(params, X_tr[b], p_drop=p_drop, rng=rng, train=True)
            grads = backward(params, acts, masks, y_tr[b])
            for i in range(len(params)):
                params[i]["W"] -= lr * (grads[i]["W"] + alpha * params[i]["W"])  # +L2 if alpha>0
                params[i]["b"] -= lr * grads[i]["b"]
    return params


print(f"{'regularizer':18} {'train':>7} {'val':>7} {'gap':>7}")
for label, kw in [("none", {}), ("dropout p=0.3", {"p_drop": 0.3}),
                  ("dropout p=0.5", {"p_drop": 0.5}), ("L2 alpha=1e-2", {"alpha": 1e-2})]:
    params = train(sizes, **kw)
    tr, va = accuracy(params, X_tr, y_tr), accuracy(params, X_val, y_val)
    print(f"{label:18} {tr:7.3f} {va:7.3f} {tr - va:+7.3f}")

In [ ]:
def train_history(p_drop=0.0, lr=0.1, epochs=1200, batch=32, every=40, seed=SEED):
    params = init_params(sizes, seed)
    rng = np.random.default_rng(seed)
    n = X_tr.shape[0]
    eps, tr_hist, val_hist = [], [], []
    for ep in range(epochs):
        order = rng.permutation(n)
        for s in range(0, n, batch):
            b = order[s:s + batch]
            acts, masks = forward(params, X_tr[b], p_drop=p_drop, rng=rng, train=True)
            grads = backward(params, acts, masks, y_tr[b])
            for i in range(len(params)):
                params[i]["W"] -= lr * grads[i]["W"]
                params[i]["b"] -= lr * grads[i]["b"]
        if ep % every == 0 or ep == epochs - 1:
            eps.append(ep)
            tr_hist.append(accuracy(params, X_tr, y_tr))
            val_hist.append(accuracy(params, X_val, y_val))
    return eps, tr_hist, val_hist


eps, tr_none, val_none = train_history(p_drop=0.0)
_, tr_drop, val_drop = train_history(p_drop=0.5)

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(eps, tr_none, color=colors.COLORS["muted"], lw=1.5, ls="--", label="train (both ~1.0)")
ax.plot(eps, val_none, color=colors.COLORS["error"], lw=2, label="validation — no dropout")
ax.plot(eps, val_drop, color=colors.COLORS["correct"], lw=2, label="validation — dropout p=0.5")
ax.set_xlabel("epoch")
ax.set_ylabel("accuracy")
ax.set_title("Train vs validation: dropout narrows the gap")
ax.legend()
plt.show()

**Read the figure.** Both networks drive **training** accuracy to 1.0 within a few epochs (grey dashed)
— they have the capacity to memorize all 200 rows. The gap is in the **validation** curves: without
dropout (coral) the held-out accuracy settles around `0.75`; with dropout (teal) it holds a couple of
points higher, around `0.77`, so the train–validation gap is visibly smaller. A real improvement — and
an honest one: it is modest, not dramatic, which is exactly what to expect on a network this size.

## Honest scope

On a small network like this, dropout is a **gentle** regularizer: here it buys two or three held-out
points, about the same as L2 (see the table above). Read that table with care — the *ordering* between
dropout and L2 (and between `p=0.3` and `p=0.5`) turns on differences of a few thousandths, well within
the noise of a single train/validation split, and it flips if you reshuffle the data. The **robust**
fact is the one that holds across splits: both regularizers beat no regularizer; which of them wins by
a hair is not something one split can tell you (the chapter-11 capstone lesson — never crown a winner
on a gap smaller than the noise).

Its **decisive** wins come on large, high-dimensional networks — the original dropout paper (Srivastava
et al. 2014) reports its biggest gains on image and speech models with millions of weights, where there
is far more co-adaptation to break. We have shown the honest, modest version on a network we can train
by hand; the mechanism is the same one that matters at scale.

## Dropout vs L2 vs early stopping

Three levers on the same disease — overfitting — with three different mechanisms:

- **L2 (`alpha`)** — add `½·alpha·Σw²` to the loss, shrinking large weights toward zero (chapter 03/11).
- **Early stopping** — watch a validation score and halt before the network memorizes (chapter 00/11).
- **Dropout** — inject multiplicative noise during training, breaking co-adaptation and averaging an
  implicit ensemble.

They are not exclusive — dropout and L2 are routinely combined. Each trades a little training fit for
better generalization; which helps most is, as always, something you **measure** on held-out data.

## What you built

- **Inverted dropout** by hand: zero a fraction `p` of activations, rescale the survivors by `1/(1−p)`
  — the mean preserved, the variance injected (`1 → 2` at `p = 0.5`).
- The **implicit-ensemble** understanding of *why* it regularizes: a different sub-network each step,
  averaged at test time, with no unit allowed to co-adapt.
- Dropout **wired into the network** — masking the forward activations *and* the backward gradient,
  active in training and off at evaluation.
- A measured, honest read of its effect (a modest held-out gain here, decisive at scale), and its place
  beside L2 and early stopping.

## Where this goes next

Dropout fights overfitting. **Notebook 6 — normalization** returns to the other thread of this chapter —
keeping the signal healthy across depth (notebooks 3–4) — but *during* training rather than only at
initialization. And when we reach PyTorch (notebook 8), all of this dropout machinery becomes a single
line, `nn.Dropout(p)`, that does exactly what you built here by hand.

## Your turn

1. **(warm-up)** Sweep `p` from 0.1 to 0.7 (re-run `train(sizes, p_drop=p)` and check validation
   accuracy). Which value generalizes best on this problem?
2. **(core)** Combine dropout **and** L2 (`train(sizes, p_drop=0.3, alpha=1e-3)`). Does the pair beat
   either one alone here, or do they step on each other?
3. **(reach)** Measure the variance of the first hidden layer's activations *with* vs *without* dropout
   at `p = 0.5` (in train mode). Does it match the `1 → 2` doubling you saw on the `N(0, 1)` batch?

## References

- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). Dropout: a
  simple way to prevent neural networks from overfitting. *Journal of Machine Learning Research*
  15:1929–1958.
- Hinton, G. E., Srivastava, N., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2012). Improving
  neural networks by preventing co-adaptation of feature detectors. arXiv:1207.0580.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*, chapter 7. MIT Press.

*Previous:* **12.4 — initialization: He and Xavier.**  *Next:* **12.6 — normalization.**